# MicroRhodoDB v1.0 — Example Usage

**MicroRhodoDB** is a standardized, research-grade database of **Type-I microbial rhodopsins**
(Bacteria, Archaea, microbial eukaryotes, viruses). Heliorhodopsins (HeR) are tracked as a
**separate companion module**, never as a Type-I subfamily.

This notebook demonstrates the Python query API. It assumes the following files sit together:

```
db/microrhododb_v1.0.sqlite      # the relational database
api/microrhododb_api.py          # this API module
```

**Key interpretation rules baked into the data:**
- Functional assignments are **inferred** (`function_is_inferred = 1` for all) from active-site
  chemistry + phylogenetic placement — they are *not* per-sequence experimental measurements.
- Spectral absorption maxima are **predicted** (`lambda_max_is_predicted = 1`) and currently emitted
  **only for proteorhodopsins** (the position-105 Leu/Gln green/blue switch). The
  `lambda_max_measured_nm` column is reserved for experimental values and is intentionally empty
  (nothing is imputed).
- `motif_agrees = 0` flags sequences where the active-site motif and the phylogenetic subfamily
  disagree; the function then follows the **motif chemistry** (conservative). These are surfaced.
- Membership is an evidence-vector classification: `confirmed_type_I`, `high_confidence_type_I`,
  `putative_type_I`, `rhodopsin_like`, `ambiguous`, `rejected`.


In [1]:
import sys, os
# point these at your local copy
API_DIR = "../api"
DB_PATH = "../db/microrhododb_v1.0.sqlite"
sys.path.insert(0, API_DIR)
from microrhododb_api import MicroRhodoDB

db = MicroRhodoDB(DB_PATH)
db.summary()

MicroRhodoDB v1.0  (../db/microrhododb_v1.0.sqlite)
  rows: protein=5361, membership=5361, subfamily=2990, function=5361, structure=4886, pdb_structure=438, predicted_structure=2, seed=15, citation=10

  membership_category:
    high_confidence_type_I   3002
    rhodopsin_like           1370
    rejected                 475
    putative_type_I          313
    ambiguous                143
    confirmed_type_I         58


,c,n
0,high_confidence_type_I,3002
1,rhodopsin_like,1370
2,rejected,475
3,putative_type_I,313
4,ambiguous,143
5,confirmed_type_I,58


## 1. What tables are in the database?

The schema is normalized around a universal key `source_acc` (UniProt accession). A denormalized
`v_master` view LEFT-JOINs every satellite table for convenient one-row-per-protein queries.

In [2]:
print("Tables & views:")
for t in db.tables():
    print("  ", t)

print("\nColumns in v_master:")
print(db.columns("v_master"))

Tables & views:
   v_master
   citation
   function
   membership
   pdb_structure
   predicted_structure
   protein
   protein_pdb
   qc_feature
   redundancy
   seed
   structure
   subfamily

Columns in v_master:
['source_acc', 'organism', 'tax_domain', 'length', 'reviewed', 'is_seed', 'membership_category', 'membership_score', 'module', 'arch_primary', 'has_retinal_lysine', 'complete_core', 'subfamily', 'subfamily_method', 'best_lwr', 'tree_discordant', 'func_class', 'func_confidence', 'helixC_motif', 'motif_agrees', 'function_is_inferred', 'spectral_class', 'lambda_max_pred_nm', 'lambda_max_is_predicted', 'lambda_max_measured_nm', 'fusion_type', 'has_experimental_structure', 'rep_pdb_id', 'rep_resolution_A', 'predicted_plddt', 'std_90_is_rep', 'backbone_is_rep']


## 2. Look up a single rhodopsin

`get_protein()` returns the full denormalized annotation for one accession. Here is
bacteriorhodopsin (the archetypal proton pump, *Halobacterium salinarum*).

In [3]:
br = db.get_protein("P02945")
br[["source_acc", "organism", "tax_domain", "membership_category", "subfamily",
    "func_class", "func_confidence", "helixC_motif", "motif_agrees",
    "has_experimental_structure", "rep_pdb_id", "rep_resolution_A"]]

source_acc                                                               P02945
organism                      Halobacterium salinarum (strain ATCC 700922 / ...
tax_domain                                                              Archaea
membership_category                                            confirmed_type_I
subfamily                                                     bacteriorhodopsin
func_class                                                          proton_pump
func_confidence                                                            high
helixC_motif                                                                DTD
motif_agrees                                                                  1
has_experimental_structure                                                    1
rep_pdb_id                                                                 5ZIM
rep_resolution_A                                                           1.25
Name: 0, dtype: object

## 3. Query by functional class or subfamily

The DTD/NDQ/NTQ active-site logic (Inoue 2015, 2020; Kandori 2015) drives ion-pump classification;
channels/sensors/enzymes are classified by architecture and phylogeny.

In [4]:
# All sodium pumps in the Type-I module
na = db.query_master(module="type_I", func_class="sodium_pump")
print("sodium pumps:", len(na))
print(na["helixC_motif"].value_counts().head())

# Functional-class composition of the whole Type-I module
typeI = db.query_master(module="type_I")
print("\nType-I functional classes:")
print(typeI["func_class"].value_counts())

sodium pumps: 81
helixC_motif
NDQ    72
NCQ     6
---     1
NDS     1
NTT     1
Name: count, dtype: int64



Type-I functional classes:
func_class
proton_pump            2252
sensory                 966
unassigned              695
anion_pump              310
light_gated_channel     267
sensory_like            152
insufficient_data        94
sodium_pump              81
enzyme_rhodopsin         69
Name: count, dtype: int64


In [5]:
# All proteorhodopsins, with their predicted spectral tuning
pr = db.subfamily("proteorhodopsin")
print("proteorhodopsins:", len(pr))
print(pr["spectral_class"].value_counts())

proteorhodopsins: 313
spectral_class
uncharacterized    175
green_predicted     79
blue_predicted      59
Name: count, dtype: int64


## 4. Spectral predictions (proteorhodopsin green/blue switch)

Only proteorhodopsins receive a predicted λmax, from the residue at aligned position 105
(Leu → green ~525 nm, Gln → blue ~490 nm). Every value is flagged predicted; none are measured.

In [6]:
sp = db.spectral_predictions()
print("predicted spectra:", len(sp), "| measured:",
      int(db.sql('SELECT COUNT(*) n FROM function WHERE lambda_max_measured_nm IS NOT NULL')["n"].iloc[0]))
sp.head(8)

predicted spectra: 138 | measured: 0


,source_acc,spectral_class,lambda_max_pred_nm,spectral_confidence,spectral_basis
0,Q9AFF7,blue_predicted,490.0,medium,proteorhodopsin position-105 = Gln -> blue-abs...
1,Q9F7P4,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...
2,A0A0B7IY85,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...
3,A0A0D6EW35,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...
4,A0A0M5L0M3,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...
5,A0A220VFA3,blue_predicted,490.0,medium,proteorhodopsin position-105 = Gln -> blue-abs...
6,A0A229FWZ5,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...
7,A0A240DXI5,green_predicted,525.0,medium,proteorhodopsin position-105 = Leu -> green-ab...


## 5. Structural annotation

66 sequences carry experimental PDB structures (438 entries; most retinal-bound). Two subfamilies
that lacked any experimental structure among their backbone representatives were modeled with
ESMFold and stored separately (predicted, not experimental).

In [7]:
# Experimental structures for bacteriorhodopsin
st = db.structures_for("P02945")
print("P02945 structures:", len(st), "| retinal-bound:", int(st["has_retinal"].sum()),
      "| best resolution:", st["resolution"].min(), "A")
display(st.head(5))

# ESMFold predicted structures
print("\nPredicted structures:")
display(db.predicted_structures())

P02945 structures: 174 | retinal-bound: 162 | best resolution: 1.05 A


,pdb_id,method,resolution,title,ligands,has_retinal,year
0,1R2N,SOLUTION NMR,NaN,NMR structure of the all-trans retinal in dark...,RET,1,2003
1,1R84,SOLUTION NMR,NaN,NMR structure of the 13-cis-15-syn retinal in ...,RET,1,2003
2,5ZIM,X-RAY DIFFRACTION,1.25,Crystal structure of bacteriorhodopsin at 1.25...,L2P;RET,1,2018
3,5ZIN,X-RAY DIFFRACTION,1.27,Crystal structure of bacteriorhodopsin at 1.27...,L2P;RET,1,2018
4,5ZIL,X-RAY DIFFRACTION,1.29,Crystal structure of bacteriorhodopsin at 1.29...,L2P;RET,1,2018



Predicted structures:


,source_acc,method,mean_plddt,pdb_file,subfamily,note
0,A0A9X2L7U0,ESMCFold2 (full),93.49,structures/A0A9X2L7U0_esmfold.pdb,chloride_pump_NTQ,ESMFold prediction for clade rep lacking exper...
1,B0R633,ESMCFold2 (full),93.71,structures/B0R633_esmfold.pdb,sensory_rhodopsin_I,ESMFold prediction for clade rep lacking exper...


## 6. Redundancy tiers (representative sets for phylogenetics / ML)

Four nested tiers: exact-NR, ~90% (`std_90`), intermediate, and the phylogenetic backbone.
Use these to pull non-redundant working sets.

In [8]:
reps = db.representatives("std_90")
print("std_90 representatives (all modules):", len(reps))
print(reps["subfamily"].value_counts().head(8))

std_90 representatives (all modules): 3262
subfamily
sensory_rhodopsin_II    546
halorhodopsin           361
anabaena_sensory        358
bacteriorhodopsin       326
proteorhodopsin         311
unclassified_typeI      307
xanthorhodopsin         277
channelrhodopsin        266
Name: count, dtype: int64


## 7. Export sequences to FASTA

Pull sequences for any accession list. Headers carry the subfamily and organism.

In [9]:
seeds = db.seeds()["acc"].tolist()
n = db.fasta(seeds, "seed_rhodopsins.faa")
print(f"wrote {n} seed sequences to seed_rhodopsins.faa")
print(open("seed_rhodopsins.faa").read()[:300])

wrote 15 seed sequences to seed_rhodopsins.faa


>A0A060H1D7 enzyme_rhodopsin | Blastocladiella emersonii (Aquatic fungus)
MKDKDNNLRGACSSCNCPEYCFSPTSTLCDDCKCSVTKHPIVEQPLSRNGSFRSSGASLLPSPSSPNVKITSTVGLRSRKSESQANVRGSMISNSNSGSRSNNSGGAGGGSGGSSSSKGGSALANYQSAMSELWSWNMMLSTPSLKFLTVQFTTWIVLTTVGAIYTLFFHERQAYNRGWADIWYGYGAFGFGLGLSFAYMGFTGARNPEKKALSLCLLGVNFISFM


## 8. Custom SQL

For anything the convenience methods don't cover, run read-only SQL directly. Example:
which taxonomic domains contribute the confirmed + high-confidence Type-I proton pumps?

In [10]:
db.sql('''
    SELECT tax_domain, COUNT(*) AS n
    FROM v_master
    WHERE func_class = 'proton_pump'
      AND membership_category IN ('confirmed_type_I', 'high_confidence_type_I')
    GROUP BY tax_domain
    ORDER BY n DESC
''')

,tax_domain,n
0,Eukaryota,945
1,Bacteria,666
2,Archaea,230


## 9. Provenance & citations

Every functional/spectral rule is backed by literature stored in the `citation` table.

In [11]:
db.citations()

,ref_id,citation,doi,context
0,1,"Mao J, et al. 2024. Structural basis of the gr...",10.1126/sciadv.adj0384,proteorhodopsin position-105 L/Q spectral tuning
1,2,"Man D, et al. 2003. Diversification and spectr...",10.1093/emboj/cdg183,proteorhodopsin L105Q spectral tuning switch (...
2,5,"Wang WW, et al. 2003. Spectral tuning in the g...",10.1074/jbc.m305716200,proteorhodopsin position-105 residue governs l...
3,7,"Ozaki Y, et al. 2014. Molecular mechanism of s...",10.1021/bi500842w,lambda_max scales with residue volume at posit...
4,8,"Kim K, et al. 2016. Crystal structure of an an...",10.1038/ncomms12677,NTQ motif anion (chloride) pump
5,9,Kandori H. 2015. Ion-pumping microbial rhodops...,10.3389/fmolb.2015.00052,ion-pump active-site residue framework
6,10,"Yun JH, et al. 2020. Non-canonical anion pump ...",10.1126/sciadv.aay2042,NTQ anion pump structural mechanism
7,11,"Besaw JE, et al. 2020. Structure of a light-dr...",10.1074/jbc.ra120.014118,NTQ anion pump
8,12,"Inoue K, et al. 2015. Convertibility between i...",10.1002/anie.201504549,NDQ motif sodium pump; DTD/DTE proton pump map...
9,13,Inoue K. 2020. Diversity and mechanism of micr...,10.1074/jbc.H120.016032,DTD/NDQ/NTQ function-associated residue determ...
